# Save Pretrained Models for Dashboard

**Purpose:** Train all 4 models once on your local machine and save them to disk.  
The dashboard on Render will load these files instead of retraining — startup time goes from ~4 minutes to ~15 seconds.

**Run this notebook once, then upload the `models/` folder to GitHub.**

### What this notebook does
1. Loads and prepares `lending_club_sample.csv`
2. Trains 4 models (RF reduced to 50 trees to keep file size under 25 MB)
3. Saves models + scaler + test data + probabilities to `models/`
4. Verifies file sizes before upload

In [1]:
import os, joblib, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score

# Create models directory
os.makedirs('models', exist_ok=True)
print('models/ folder ready')

models/ folder ready


## 1. Configuration
These must match the values in `dashboard.py` exactly.

In [2]:
CSV_FILE     = 'lending_club_sample.csv'
NROWS        = None   # None = use all rows for best model quality
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# RF reduced to 50 trees to keep pkl under 25 MB (GitHub limit)
RF_N_ESTIMATORS = 50

## 2. Load & Prepare Data

In [3]:
print(f'Loading {CSV_FILE}...')
df = pd.read_csv(CSV_FILE, nrows=NROWS, low_memory=False)

# Imputation
for col in ['revol_util', 'bc_util', 'pct_tl_nvr_dlq', 'percent_bc_gt_75']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())
for col in ['mort_acc', 'pub_rec_bankruptcies', 'num_accts_ever_120_pd']:
    if col in df.columns:
        df[col] = df[col].fillna(0)
if 'emp_length' in df.columns:
    df['emp_length'] = df['emp_length'].fillna(df['emp_length'].mode()[0])

# Target
default_statuses = ['Charged Off','Late (16-30 days)','Late (31-120 days)','In Grace Period','Default']
df['default'] = df['loan_status'].isin(default_statuses).astype(int)

# Feature engineering
if 'grade_num' not in df.columns:
    df['grade_num'] = df['grade'].map({'A':1,'B':2,'C':3,'D':4,'E':5,'F':6,'G':7})
if 'term_months' not in df.columns:
    df['term_months'] = df['term'].str.replace(' months','').str.strip().astype(float)
if 'emp_length_yrs' not in df.columns:
    el = df['emp_length'].astype(str)
    el = el.str.replace('< 1 year','0').str.replace('10+ years','10')
    el = el.str.replace(' years','').str.replace(' year','')
    df['emp_length_yrs'] = pd.to_numeric(el, errors='coerce').fillna(0)
if 'fico_avg' not in df.columns:
    df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2

df['installment_to_income'] = df['installment'] / (df['annual_inc'].clip(lower=1) / 12)
if 'earliest_cr_line' in df.columns:
    df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], errors='coerce')
    df['credit_history_years'] = (pd.Timestamp('2018-12-31') - df['earliest_cr_line']).dt.days / 365.25
else:
    df['credit_history_years'] = 10.0
df['delinquency_flag'] = (
    (df.get('delinq_2yrs', 0) > 0) | (df.get('num_accts_ever_120_pd', 0) > 0)
).astype(int)

print(f'  Rows: {len(df):,}  |  Default rate: {df["default"].mean():.1%}')

Loading lending_club_sample.csv...
  Rows: 102,777  |  Default rate: 19.4%


## 3. Feature Matrix

In [4]:
df_model = df.copy()
df_model = pd.get_dummies(
    df_model,
    columns=['home_ownership','purpose','verification_status','application_type'],
    drop_first=True
)
cols_to_drop = ['loan_status','default_label','grade','sub_grade','term','emp_length',
                'addr_state','earliest_cr_line','fico_range_low','fico_range_high',
                'issue_d']
cols_to_drop = [c for c in cols_to_drop if c in df_model.columns]

y = df_model['default']
X = df_model.drop(columns=cols_to_drop + ['default'])
X = X.select_dtypes(include=[np.number])
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}  |  Features: {X.shape[1]}')

Train: 82,221  |  Test: 20,556  |  Features: 27


## 4. Train Models

In [5]:
print('Training Logistic Regression...')
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_sc, y_train)
print(f'  AUC: {roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:,1]):.4f}')

Training Logistic Regression...
  AUC: 0.7046


In [6]:
print('Training Decision Tree...')
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=50,
     class_weight='balanced', random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
print(f'  AUC: {roc_auc_score(y_test, dt.predict_proba(X_test)[:,1]):.4f}')

Training Decision Tree...
  AUC: 0.7039


In [7]:
print(f'Training Random Forest ({RF_N_ESTIMATORS} trees)...')
rf = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_features='sqrt', max_depth=15, min_samples_leaf=20,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
print(f'  AUC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}')

Training Random Forest (50 trees)...
  AUC: 0.7198


In [8]:
print('Training XGBoost...')
neg, pos = (y_train==0).sum(), (y_train==1).sum()
xgb = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=round(neg/pos, 2),
    eval_metric='logloss', random_state=RANDOM_STATE)
xgb.fit(X_train, y_train)
print(f'  AUC: {roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]):.4f}')

Training XGBoost...
  AUC: 0.7272


## 4b. Train LightGBM
LightGBM — histogram-based gradient boosting, faster than XGBoost.

In [9]:
from lightgbm import LGBMClassifier

print('Training LightGBM...')
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)
lgbm.fit(X_train, y_train)
print(f'  AUC: {roc_auc_score(y_test, lgbm.predict_proba(X_test)[:,1]):.4f}')

Training LightGBM...
  AUC: 0.7270


## 5. Compute Probabilities & Optimal Threshold

In [10]:
proba_lr  = lr.predict_proba(X_test_sc)[:,1]
proba_dt  = dt.predict_proba(X_test)[:,1]
proba_rf  = rf.predict_proba(X_test)[:,1]
proba_xgb = xgb.predict_proba(X_test)[:,1]

# Optimal threshold (max F1) for XGBoost
thresholds = np.arange(0.05, 0.95, 0.01)
f1_scores  = [f1_score(y_test, (proba_xgb >= t).astype(int), pos_label=1) for t in thresholds]
optimal_threshold = round(float(thresholds[np.argmax(f1_scores)]), 2)
print(f'Optimal threshold (max F1): {optimal_threshold}')

Optimal threshold (max F1): 0.5


## 5. 5-Fold Cross-Validation
More reliable performance estimates across all 5 models.

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

print('Running 5-fold cross-validation...')
cv_models = [
    ('Logistic Regression', lr, X_train_sc),
    ('Decision Tree',       dt, X_train),
    ('Random Forest',       rf, X_train),
    ('XGBoost',            xgb, X_train),
    ('LightGBM',          lgbm, X_train),
]

cv_results = []
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model, X_cv in cv_models:
    scores = cross_val_score(model, X_cv, y_train,
                             cv=kf, scoring='roc_auc', n_jobs=-1)
    cv_results.append({
        'model':    name,
        'mean_auc': round(float(scores.mean()), 4),
        'std_auc':  round(float(scores.std()),  4),
        'gini':     round(float(2*scores.mean()-1), 4),
        'min_auc':  round(float(scores.min()), 4),
        'max_auc':  round(float(scores.max()), 4),
    })
    print(f'  {name:<25} AUC={scores.mean():.4f} (±{scores.std():.4f})')

print('CV done.')

Running 5-fold cross-validation...
  Logistic Regression       AUC=0.7098 (±0.0026)
  Decision Tree             AUC=0.7052 (±0.0030)
  Random Forest             AUC=0.7224 (±0.0018)
  XGBoost                   AUC=0.7303 (±0.0025)
  LightGBM                  AUC=0.7298 (±0.0027)
CV done.


## 6. Save Everything to `models/`

In [15]:
print('Saving models...')

joblib.dump(lr,     'models/lr.pkl')
joblib.dump(dt,     'models/dt.pkl')
joblib.dump(rf,     'models/rf.pkl')
joblib.dump(xgb,    'models/xgb.pkl')
joblib.dump(lgbm,   'models/lgbm.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

# Save test data and probabilities so dashboard can load them directly
joblib.dump({
    'X_test':           X_test,
    'y_test':           y_test,
    'X_train_medians':  X_train.median().to_dict(),
    'feature_cols':     list(X.columns),
    'proba_lr':         proba_lr,
    'proba_dt':         proba_dt,
    'proba_rf':         proba_rf,
    'proba_xgb':        proba_xgb,
    'proba_lgbm':        lgbm.predict_proba(X_test)[:,1],
    'optimal_threshold': optimal_threshold,
    'cv_results':        cv_results,
    'X_shap':            X_shap,
    'thresholds':       thresholds,
    'f1_scores':        f1_scores,
}, 'models/eval_data.pkl')

print('All files saved to models/')

Saving models...
All files saved to models/


## 7. Verify File Sizes
All files should be under 25 MB to stay within GitHub's recommended limit.

## 7. Generate & Save SHAP Plots
These 4 PNGs are fixed — generated once, loaded by the dashboard directly.

In [16]:
import shap, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Computing SHAP values (500 samples)...')
np.random.seed(42)
shap_idx    = np.random.choice(len(X_test), size=500, replace=False)
X_shap      = X_test.iloc[shap_idx].reset_index(drop=True)
explainer   = shap.TreeExplainer(xgb.get_booster())
shap_values = explainer(X_shap)

proba_shap    = xgb.predict_proba(X_shap)[:,1]
high_risk_idx = int(np.argsort(proba_shap)[-1])
low_risk_idx  = int(np.argsort(proba_shap)[0])

def save_fig(path):
    plt.gcf().savefig(path, bbox_inches='tight', dpi=72)
    plt.close('all')
    print(f'  Saved: {path}  ({os.path.getsize(path)/1024:.0f} KB)')

plt.figure(figsize=(10,7))
shap.summary_plot(shap_values.values, X_shap, max_display=15, show=False)
plt.title('SHAP Summary — Global Feature Impact & Direction', fontweight='bold')
plt.tight_layout()
save_fig('models/shap_beeswarm.png')

plt.figure(figsize=(9,6))
shap.summary_plot(shap_values.values, X_shap, plot_type='bar', max_display=15, show=False)
plt.title('SHAP Feature Importance (mean |SHAP value|)', fontweight='bold')
plt.tight_layout()
save_fig('models/shap_bar.png')

plt.figure(figsize=(10,6))
shap.plots.waterfall(shap_values[high_risk_idx], max_display=12, show=False)
plt.title(f'High-Risk Borrower  (p = {proba_shap[high_risk_idx]:.1%})', fontweight='bold')
plt.tight_layout()
save_fig('models/shap_waterfall_high.png')

plt.figure(figsize=(10,6))
shap.plots.waterfall(shap_values[low_risk_idx], max_display=12, show=False)
plt.title(f'Low-Risk Borrower  (p = {proba_shap[low_risk_idx]:.1%})', fontweight='bold')
plt.tight_layout()
save_fig('models/shap_waterfall_low.png')

print('All SHAP plots saved.')

Computing SHAP values (500 samples)...
  Saved: models/shap_beeswarm.png  (54 KB)
  Saved: models/shap_bar.png  (26 KB)
  Saved: models/shap_waterfall_high.png  (39 KB)
  Saved: models/shap_waterfall_low.png  (41 KB)
All SHAP plots saved.


In [17]:
print('File sizes in models/:')
print('-' * 35)
total = 0
for fname in sorted(os.listdir('models')):
    size_mb = os.path.getsize(f'models/{fname}') / 1024 / 1024
    total += size_mb
    flag = '  OK' if size_mb < 25 else '  WARNING: too large for GitHub'
    print(f'  {fname:<25} {size_mb:>6.1f} MB{flag}')
print('-' * 35)
print(f'  Total: {total:.1f} MB')

if total < 100:
    print('\nAll good — safe to upload to GitHub.')
else:
    print('\nTotal exceeds 100 MB — consider reducing RF n_estimators further.')

File sizes in models/:
-----------------------------------
  dt.pkl                       0.0 MB  OK
  eval_data.pkl                5.8 MB  OK
  lgbm.pkl                     0.5 MB  OK
  lr.pkl                       0.0 MB  OK
  rf.pkl                      10.5 MB  OK
  scaler.pkl                   0.0 MB  OK
  shap_bar.png                 0.0 MB  OK
  shap_beeswarm.png            0.1 MB  OK
  shap_waterfall_high.png      0.0 MB  OK
  shap_waterfall_low.png       0.0 MB  OK
  xgb.pkl                      0.5 MB  OK
-----------------------------------
  Total: 17.5 MB

All good — safe to upload to GitHub.


## 8. Next Steps

1. **Upload `models/` folder to GitHub** — drag and drop all `.pkl` files into the repo
2. **Update `dashboard.py`** — replace the training section with `joblib.load()` calls
3. **Redeploy on Render** — startup will go from ~4 minutes to ~15 seconds

The dashboard will automatically use these pre-trained models.  
If you retrain with new data or parameters, just re-run this notebook and re-upload.